<a href="https://colab.research.google.com/github/Mariano-rr/ThinkPythonAssignments/blob/main/Week15.5/NutritionToolkit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pandas
import pandas as pd
import json
import os
from datetime import datetime, timedelta

class NutritionToolkit:
    def __init__(self):
        self.macro_file = "macro_log.csv"
        self.fridge_file = "fridge_log.csv"
        self.recipe_file = "recipes.json"
        # Logic: Internal goals mapped to CSV column names
        self.goals = {'p_grams': 120, 'c_grams': 200, 'f_grams': 70}
        self._init_files()

    def _init_files(self):
        """Initializes storage with protection against corruption and locks."""
        try:
            if not os.path.exists(self.macro_file) or os.path.getsize(self.macro_file) == 0:
                pd.DataFrame(columns=['date', 'food', 'p_grams', 'c_grams', 'f_grams', 'calories']).to_csv(self.macro_file, index=False)
            if not os.path.exists(self.fridge_file) or os.path.getsize(self.fridge_file) == 0:
                pd.DataFrame(columns=['item', 'cooked_date', 'expiry_date']).to_csv(self.fridge_file, index=False)
            if not os.path.exists(self.recipe_file):
                with open(self.recipe_file, 'w') as f:
                    json.dump({}, f)
        except PermissionError:
            print("\n[!] ACCESS DENIED: Close your CSV files in Excel and restart.")
            exit() # Stop the program safely if it can't reach its data

    def view_today_progress(self):
        """Robust Progress Tracking with error handling for empty/bad data."""
        try:
            df = pd.read_csv(self.macro_file)
            today = str(datetime.now().date())
            todays_data = df[df['date'] == today]

            if todays_data.empty:
                print(f"\n[!] No meals logged for today ({today}).")
                return

            totals = todays_data[['p_grams', 'c_grams', 'f_grams', 'calories']].sum()

            print(f"\n--- PROGRESS FOR {today} ---")
            for key, label in [('p_grams', 'Protein'), ('c_grams', 'Carbs'), ('f_grams', 'Fat')]:
                current = totals[key]
                goal = self.goals[key]
                percent = (current / goal) * 100 if goal > 0 else 0
                print(f"{label:<8}: {current:>5.1f}g / {goal:>3}g ({percent:>5.1f}%)")
            print(f"CALORIES: {totals['calories']:.0f} kcal")
            print("-" * 30)
        except Exception as e:
            print(f"\n[!] Error reading progress: {e}. Your CSV file might be corrupted.")

    def log_meal(self, name, p, c, f):
        try:
            calories = (p * 4) + (c * 4) + (f * 9)
            new_row = pd.DataFrame([[str(datetime.now().date()), name, p, c, f, calories]],
                                   columns=['date', 'food', 'p_grams', 'c_grams', 'f_grams', 'calories'])
            new_row.to_csv(self.macro_file, mode='a', header=False, index=False)
            print(f"\n[✓] Logged: {name}")
        except Exception as e:
            print(f"\n[!] Error saving meal: {e}")

    def add_leftover(self, item_name):
        expires = datetime.now().date() + timedelta(days=4)
        new_item = pd.DataFrame([[item_name, str(datetime.now().date()), str(expires)]],
                                columns=['item', 'cooked_date', 'expiry_date'])
        new_item.to_csv(self.fridge_file, mode='a', header=False, index=False)
        print(f"\n[!] Added to fridge. Safe until {expires}.")

    def check_fridge(self):
        try:
            df = pd.read_csv(self.fridge_file)
            if df.empty: print("\nFridge is empty!"); return
            df['expiry_date'] = pd.to_datetime(df['expiry_date']).dt.date
            safe_items = df[df['expiry_date'] >= datetime.now().date()]
            print(f"\n--- FRIDGE STATUS ---")
            print(safe_items[['item', 'expiry_date']].to_string(index=False) if not safe_items.empty else "All items expired.")
        except Exception as e:
            print(f"\n[!] Error checking fridge: {e}")

    def save_recipe(self, name, servings, ingredients):
        try:
            with open(self.recipe_file, 'r') as f:
                recipes = json.load(f)
            recipes[name.lower()] = {"display_name": name, "servings": servings, "ingredients": ingredients}
            with open(self.recipe_file, 'w') as f:
                json.dump(recipes, f, indent=4)
            print(f"\n[✓] Recipe '{name}' saved.")
        except json.JSONDecodeError:
            print("\n[!] Recipe file is corrupted. Try deleting 'recipes.json' and restarting.")

    def list_recipes(self):
        try:
            with open(self.recipe_file, 'r') as f:
                recipes = json.load(f)
            if not recipes:
                print("\n[!] No recipes found."); return False
            print("\n--- SAVED RECIPES ---")
            for r in recipes.values():
                print(f"- {r['display_name']} ({r['servings']} servings)")
            return True
        except: return False

    def scale_and_export(self, recipe_name, target_servings):
        try:
            with open(self.recipe_file, 'r') as f:
                recipes = json.load(f)
            key = recipe_name.lower()
            if key not in recipes:
                print(f"\n[X] Error: '{recipe_name}' not found."); return

            data = recipes[key]
            if data['servings'] <= 0: print("\n[!] Error: Recipe has 0 servings."); return

            factor = target_servings / data['servings']
            total_cost = 0
            safe_fname = "".join(x for x in data['display_name'] if x.isalnum() or x in " -_").strip().lower()
            filename = f"shopping_list_{safe_fname}.txt"

            with open(filename, "w") as f:
                f.write(f"SHOPPING LIST: {data['display_name']} ({target_servings} Servings)\n" + "="*60 + "\n")
                f.write(f"{'Item':<18} | {'Qty':>12} | {'Cost':>12}\n" + "-"*60 + "\n")
                for item, details in data['ingredients'].items():
                    amt, unit_price = details
                    scaled_amt, cost = amt * factor, amt * factor * unit_price
                    total_cost += cost
                    f.write(f"{item:<18} | {scaled_amt:>12.2f} | ${cost:>11.2f}\n")
                f.write("="*60 + f"\nTOTAL ESTIMATED COST: ${total_cost:.2f}\n")
            print(f"\n[✓] List exported to: {filename}")
        except Exception as e:
            print(f"\n[!] Scaling failed: {e}")

# --- GLOBAL SAFETY HELPERS ---
def get_valid_name(prompt):
    while True:
        val = input(prompt).strip()
        if not val: print("Error: Blank names not allowed."); continue
        if val.isdigit(): print("Error: Use words, not numbers."); continue
        return val.title()

def get_valid_num(prompt, type_func=float, min_val=0):
    while True:
        raw = input(prompt).strip()
        try:
            val = type_func(raw)
            if val < min_val: print(f"Error: Must be at least {min_val}."); continue
            return val
        except ValueError: print("Error: Enter a valid number.")

def main():
    kit = NutritionToolkit()
    while True:
        print("\n=== THE UNBREAKABLE NUTRITION TOOL ===")
        print("1: Log Meal        | 2: View Progress")
        print("3: Fridge Status   | 4: Save Recipe")
        print("5: Scale Recipe    | 6: Exit")
        cmd = input("Choice: ").strip()

        try:
            if cmd == "1":
                kit.log_meal(get_valid_name("Food Name: "), get_valid_num("P(g): "), get_valid_num("C(g): "), get_valid_num("F(g): "))
            elif cmd == "2": kit.view_today_progress()
            elif cmd == "3":
                if input("(A)dd or (V)iew? ").lower().strip() == 'a': kit.add_leftover(get_valid_name("Item: "))
                else: kit.check_fridge()
            elif cmd == "4":
                name = get_valid_name("Recipe Name: ")
                servings = get_valid_num("Base Servings: ", int, min_val=1)
                ingredients = {}
                while True:
                    ing = input("\nIngredient Name (or 'DONE'): ").strip()
                    if ing.upper() == 'DONE':
                        if not ingredients: print("Add an ingredient first!"); continue
                        break
                    if not ing or ing.isdigit(): print("Invalid name."); continue
                    amt = get_valid_num(f"Amount: ", min_val=0.01)
                    price = get_valid_num(f"Price per unit: $")
                    ingredients[ing.title()] = [amt, price]
                kit.save_recipe(name, servings, ingredients)
            elif cmd == "5":
                if kit.list_recipes():
                    target = input("\nRecipe to scale: ").strip()
                    if target: kit.scale_and_export(target, get_valid_num("Target Servings: ", int, min_val=1))
            elif cmd == "6": break
        except KeyboardInterrupt:
            print("\nExiting safely...")
            break

if __name__ == "__main__":
    main()



